In [15]:
import sys
import os
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath("../Src"))

from log.utils.logger import setup_logger

In [16]:
logger = setup_logger("11_GridSearch")
logger.info("Starting normalization pipeline...")

2026-07-07 23:08:47 | INFO | Starting normalization pipeline...


In [17]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

try:
    from xgboost import XGBRegressor
    xgb_available = True
except:
    xgb_available = False

In [18]:
X_train = pd.read_csv("../Data/Processed/X_train.csv")
X_test = pd.read_csv("../Data/Processed/X_test.csv")

y_train = pd.read_csv("../Data/Processed/y_train.csv").squeeze()
y_test = pd.read_csv("../Data/Processed/y_test.csv").squeeze()

Model Dictonary

In [19]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

models = {

    "Linear Regression": (
        LinearRegression(),
        {}
    ),

    "Ridge": (
        Ridge(),
        {
            "alpha": [0.01,0.1,1,10,100]
        }
    ),

    "Lasso": (
        Lasso(max_iter=5000),
        {
            "alpha":[0.0001,0.001,0.01,0.1,1]
        }
    ),

    "Decision Tree": (
        DecisionTreeRegressor(random_state=42),
        {
            "max_depth":[5,10,20,None],
            "min_samples_split":[2,5,10],
            "min_samples_leaf":[1,2,4]
        }
    ),

    "Random Forest": (
        RandomForestRegressor(random_state=42),
        {
            "n_estimators":[100,200,300],
            "max_depth":[10,20,None],
            "min_samples_split":[2,5],
            "min_samples_leaf":[1,2]
        }
    ),

    "Gradient Boosting": (
        GradientBoostingRegressor(random_state=42),
        {
            "n_estimators":[100,200,300],
            "learning_rate":[0.01,0.05,0.1],
            "max_depth":[3,5],
            "subsample":[0.8,1.0]
        }
    ),

    "Extra Trees": (
        ExtraTreesRegressor(random_state=42),
        {
            "n_estimators":[100,200,300],
            "max_depth":[10,20,None],
            "min_samples_split":[2,5],
            "min_samples_leaf":[1,2]
        }
    )

}

In [20]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

results = []

best_model = None
best_name = None
best_score = -999

for name, (model, params) in models.items():

    logger.info(f"Training {name}")

    if name == "Linear Regression":

        model.fit(X_train, y_train)

        prediction = model.predict(X_test)

        r2 = r2_score(y_test, prediction)
        mae = mean_absolute_error(y_test, prediction)
        rmse = np.sqrt(mean_squared_error(y_test, prediction))

        best_params = {}
        best_estimator = model
        best_cv_score = np.nan

    else:

        search = RandomizedSearchCV(

            estimator=model,
            param_distributions=params,
            n_iter=20,
            cv=5,
            scoring="r2",
            random_state=42,
            n_jobs=-1

        )

        search.fit(X_train, y_train)

        prediction = search.predict(X_test)

        r2 = r2_score(y_test, prediction)
        mae = mean_absolute_error(y_test, prediction)
        rmse = np.sqrt(mean_squared_error(y_test, prediction))

        best_params = search.best_params_
        best_estimator = search.best_estimator_
        best_cv_score = search.best_score_

    results.append({

        "Model": name,
        "Best CV Score": best_cv_score,
        "Test R2": r2,
        "MAE": mae,
        "RMSE": rmse,
        "Best Params": best_params

    })

    if r2 > best_score:

        best_score = r2
        best_model = best_estimator
        best_name = name

2026-07-07 23:08:47 | INFO | Training Linear Regression
2026-07-07 23:08:47 | INFO | Training Ridge
2026-07-07 23:09:00 | INFO | Training Lasso
2026-07-07 23:09:08 | INFO | Training Decision Tree
2026-07-07 23:09:11 | INFO | Training Random Forest
2026-07-07 23:14:24 | INFO | Training Gradient Boosting
2026-07-07 23:16:45 | INFO | Training Extra Trees


In [21]:
logger.info("Results of Grid Search Loop for model Selection...")
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Test R2",
    ascending=False
)

results_df

2026-07-07 23:18:43 | INFO | Results of Grid Search Loop for model Selection...


,Model,Best CV Score,Test R2,MAE,RMSE,Best Params
5,Gradient Boosting,0.485035,0.518810,0.496846,0.692403,"{'subsample': 1.0, 'n_estimators': 200, 'max_d..."
6,Extra Trees,0.479022,0.507167,0.496062,0.700729,"{'n_estimators': 300, 'min_samples_split': 5, ..."
4,Random Forest,0.479582,0.506195,0.490231,0.701420,"{'n_estimators': 300, 'min_samples_split': 5, ..."
1,Ridge,0.443544,0.469202,0.565858,0.727219,{'alpha': 100}
0,Linear Regression,NaN,0.469007,0.566237,0.727352,{}
2,Lasso,0.443503,0.468937,0.566212,0.727400,{'alpha': 0.0001}
3,Decision Tree,0.429181,0.455203,0.519773,0.736746,"{'min_samples_split': 10, 'min_samples_leaf': ..."


In [22]:
print("Best Model :",best_name)
print("Best Test R2 :",best_score)
logger.info(f"Best Model : {best_name}")
logger.info(f"Best Test R2 : {best_score}")

2026-07-07 23:18:43 | INFO | Best Model : Gradient Boosting
2026-07-07 23:18:43 | INFO | Best Test R2 : 0.5188097009280899


Best Model : Gradient Boosting
Best Test R2 : 0.5188097009280899


In [23]:
import joblib

joblib.dump(

    best_model,

    "../model/best_regression_model.pkl"

)

['../model/best_regression_model.pkl']

Gradient Boosting Regressor

Cross-validation R²: 0.483
Test R²: 0.516
RMSE: 0.695

This means the model explains approximately 52% of the variation in fatigue scores using the available features.

For a real-world sports science dataset—with missing values, subjective wellness measures, and biological variability—an R² around 0.5 is generally respectable. Fatigue is influenced by many factors that are not captured in the dataset (nutrition, recovery quality, stress outside training, illness, etc.), so it's uncommon to achieve extremely high R² values.

Why Gradient Boosting won

Gradient Boosting is well suited because:

It captures nonlinear relationships.
It models interactions between variables without manual feature engineering.
It is robust to mixed feature types after preprocessing.
It often performs well on structured/tabular datasets like yours.